In [2]:
# Install required libraries
!pip install scipy -i https://pypi.tuna.tsinghua.edu.cn/simple
!pip install scikit-learn joblib

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
  Using cached scikit_learn-1.9.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (9.3 kB)
  Using cached narwhals-2.28.0-py3-none-any.whl.metadata (15 kB)
  Using cached threadpoolctl-3.8.0-py3-none-any.whl.metadata (29 kB)
Using cached scikit_learn-1.9.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (9.2 MB)
Using cached narwhals-2.28.0-py3-none-any.whl (490 kB)
Using cached threadpoolctl-3.8.0-py3-none-any.whl (32 kB)


In [3]:
import pandas as pd
import numpy as np
import urllib.request

# 1. Download and parse ARFF file
url = "https://openml.org/data/v1/download/1586239/dresses-sales.arff"
response = urllib.request.urlopen(url)
lines = response.read().decode("utf-8").splitlines()

columns = []
data_start = None
for i, line in enumerate(lines):
    line = line.strip()
    if line.lower().startswith("@attribute"):
        rest = line[len("@attribute"):].strip()
        if rest.startswith("'"):
            end = rest.index("'", 1)
            col_name = rest[1:end]
        elif rest.startswith('"'):
            end = rest.index('"', 1)
            col_name = rest[1:end]
        else:
            col_name = rest.split()[0]
        columns.append(col_name)
    if line.lower() == "@data":
        data_start = i + 1
        break

rows = []
for line in lines[data_start:]:
    line = line.strip()
    if not line:
        continue
    rows.append([v.strip().strip("'\"") for v in line.split(",")])

df = pd.DataFrame(rows, columns=columns)

# 2. Rename columns to readable names
col_names = ['Style', 'Price', 'Rating', 'Size', 'Season', 'NeckLine',
             'SleeveLength', 'waiseline', 'Material', 'FabricType',
             'Decoration', 'Pattern Type', 'Recommendation']
df.columns = col_names

# 3. Handle "?" missing values
df = df.replace("?", np.nan)
for col in df.columns:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

print("Dataset Shape:", df.shape)
print(df.head(2))

Dataset Shape: (500, 13)
    Style Price Rating Size  Season NeckLine SleeveLength waiseline  \
0    Sexy   Low    4.6    M  Summer   o-neck    sleevless    empire   
1  Casual   Low      0    L  Summer   o-neck        Petal   natural   

     Material FabricType Decoration Pattern Type Recommendation  
0      cotton    chiffon    ruffles       animal              2  
1  microfiber    chiffon    ruffles       animal              1  


In [4]:
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

# 1. Map target 1/2 -> 0/1
df["Recommendation"] = df["Recommendation"].astype(int).map({1: 0, 2: 1})

# 2. Encode categorical features
le_dict = {}
for col in df.columns:
    if df[col].dtype == "object":
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
        le_dict[col] = le

# 3. Split features and target
target_col = "Recommendation"
X = df.drop(columns=[target_col])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Data split and scaled successfully.")

Data split and scaled successfully.


In [5]:
import os
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,
                             recall_score, f1_score, matthews_corrcoef)

# 1. Train models and calculate metrics
os.makedirs("model", exist_ok=True)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes": GaussianNB(),
    "SVM": SVC(probability=True, random_state=42)
}

results = {}
for name, model in models.items():
    if name in ["Logistic Regression", "KNN", "SVM"]:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]

    results[name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "AUC": roc_auc_score(y_test, y_prob),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "MCC": matthews_corrcoef(y_test, y_pred)
    }
    joblib.dump(model, f"model/{name.replace(' ', '_')}.pkl")

# 2. Output comparison table
result_df = pd.DataFrame(results).T.round(4)
print("\n--- Model Evaluation Results ---")
print(result_df)
result_df.to_csv("model_results.csv")

# 3. Save scaler and test data for Streamlit App
joblib.dump(scaler, "model/scaler.pkl")
joblib.dump(le_dict, "model/label_encoders.pkl")

test_df = X_test.copy()
test_df[target_col] = y_test.values
test_df.to_csv("test_data.csv", index=False)
print("\ntest_data.csv saved with", len(test_df), "rows")


--- Model Evaluation Results ---
                     Accuracy     AUC  Precision  Recall      F1     MCC
Logistic Regression      0.57  0.5378     0.4800  0.2857  0.3582  0.0702
Decision Tree            0.58  0.5722     0.5000  0.5238  0.5116  0.1437
KNN                      0.54  0.5025     0.4474  0.4048  0.4250  0.0434
Naive Bayes              0.56  0.5402     0.4615  0.2857  0.3529  0.0499
SVM                      0.54  0.5337     0.4375  0.3333  0.3784  0.0243

test_data.csv saved with 100 rows


/home/cloud/anaconda3/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
